# A4, Evidence-Only, and Known-Trial Report Alignment

This notebook compares the three diagnostic report-generation conditions. It loads the existing LLM-judge outputs, verifies that the same reviews are present in every condition, builds a compact summary table, and generates the four-panel figure.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

REPOSITORY_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'gpt_researcher').is_dir()),
    None,
)
if REPOSITORY_ROOT is None:
    raise FileNotFoundError('Run this notebook from inside the repository.')

ROOT = REPOSITORY_ROOT / 'outputs' / 'evaluation' / 'report_alignment'
OUTPUT_DIR = ROOT / 'three_condition_analysis'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONDITIONS = {
    'A4': 'judge_alignment',
    'Evidence-only': 'judge_alignment_evidence_only',
    'Known-trial': 'judge_alignment_known_trials',
}
CONDITION_ORDER = list(CONDITIONS)
MODEL_ORDER = ['Qwen3-8B', 'Fleming-R1-7B']

ROOT, OUTPUT_DIR

## Load and validate the three conditions

In [ ]:
review_frames = []
keypoint_frames = []

for condition, stem in CONDITIONS.items():
    review_path = ROOT / 'outputs' / f'{stem}_review.csv'
    keypoint_path = ROOT / 'outputs' / f'{stem}_keypoints.csv'

    if not review_path.exists() or not keypoint_path.exists():
        raise FileNotFoundError(
            f'Missing input for {condition}: {review_path} or {keypoint_path}'
        )

    review_df = pd.read_csv(review_path)
    keypoint_df = pd.read_csv(keypoint_path)
    review_df['condition'] = condition
    keypoint_df['condition'] = condition
    review_frames.append(review_df)
    keypoint_frames.append(keypoint_df)

reviews = pd.concat(review_frames, ignore_index=True)
keypoints = pd.concat(keypoint_frames, ignore_index=True)

assert not reviews.duplicated(['condition', 'subject_model', 'pmid']).any()
assert not keypoints.duplicated(
    ['condition', 'subject_model', 'pmid', 'key_point_id']
).any()

expected_pmids = None
for condition in CONDITION_ORDER:
    for model in MODEL_ORDER:
        pmids = set(
            reviews.loc[
                (reviews['condition'] == condition)
                & (reviews['subject_model'] == model),
                'pmid',
            ]
        )
        assert len(pmids) == 51, (condition, model, len(pmids))
        if expected_pmids is None:
            expected_pmids = pmids
        else:
            assert pmids == expected_pmids, (condition, model)

validation = (
    reviews.groupby(['condition', 'subject_model'], observed=True)
    .agg(reviews=('pmid', 'nunique'))
    .reindex(pd.MultiIndex.from_product(
        [CONDITION_ORDER, MODEL_ORDER],
        names=['condition', 'subject_model'],
    ))
)
display(validation)
print('Validation passed: all conditions contain the same 51 PMIDs per model.')

## Aggregate semantic metrics

KPA, Core KPA, and contradiction rate are macro-averaged across reviews. Conclusion and uncertainty values are proportions of the 51 reports in each model-condition combination.

In [ ]:
def summarise_group(group):
    return pd.Series({
        'Question relevance (direct)': group['question_relevance'].eq('direct').mean(),
        'KPA': group['key_point_alignment'].mean(),
        'Core KPA': group['core_alignment'].mean(),
        'Contradiction rate': group['key_point_contradiction_rate'].mean(),
        'Aligned or partially aligned': group['conclusion_alignment'].isin(
            ['aligned', 'partially_aligned']
        ).mean(),
        'Conclusion contradicted': group['conclusion_alignment'].eq(
            'contradicted'
        ).mean(),
        'Uncertainty matched': group['uncertainty_alignment'].eq('matched').mean(),
        'Stronger than reference': group['uncertainty_alignment'].eq(
            'stronger_than_reference'
        ).mean(),
    })

summary = (
    reviews.groupby(['subject_model', 'condition'], observed=True)
    .apply(summarise_group, include_groups=False)
    .reset_index()
)
summary['subject_model'] = pd.Categorical(
    summary['subject_model'], categories=MODEL_ORDER, ordered=True
)
summary['condition'] = pd.Categorical(
    summary['condition'], categories=CONDITION_ORDER, ordered=True
)
summary = summary.sort_values(['subject_model', 'condition']).reset_index(drop=True)
summary.to_csv(OUTPUT_DIR / 'three_condition_semantic_summary.csv', index=False)

display(
    summary.style
    .format({column: '{:.3f}' for column in summary.columns[2:]})
    .hide(axis='index')
)

## Figures

All panels use a 0--100 scale to avoid visually exaggerating small changes. Lines connect diagnostic conditions only to guide the eye. They do not imply that each comparison isolates a single causal intervention.

In [ ]:
plot_metrics = [
    ('Core KPA', 'Core key-point alignment', 'Macro score × 100'),
    ('Contradiction rate', 'Key-point contradiction rate', 'Macro rate (%)'),
    (
        'Aligned or partially aligned',
        'Aligned or partially aligned conclusions',
        'Reports (%)',
    ),
    (
        'Stronger than reference',
        'Stronger-than-reference uncertainty',
        'Reports (%)',
    ),
]

styles = {
    'Qwen3-8B': {'color': '#0072B2', 'marker': 'o'},
    'Fleming-R1-7B': {'color': '#D55E00', 'marker': 's'},
}

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.labelsize': 9,
    'legend.fontsize': 9,
})

fig, axes = plt.subplots(2, 2, figsize=(10.5, 7.0), sharex=True)
fig.subplots_adjust(top=0.88, hspace=0.35, wspace=0.25)
x = np.arange(len(CONDITION_ORDER))

for ax, (column, title, ylabel), panel in zip(
    axes.flat, plot_metrics, ['a', 'b', 'c', 'd']
):
    for model in MODEL_ORDER:
        model_rows = (
            summary.loc[summary['subject_model'] == model]
            .set_index('condition')
            .reindex(CONDITION_ORDER)
        )
        values = model_rows[column].astype(float).to_numpy() * 100
        style = styles[model]

        ax.plot(
            x, values,
            color=style['color'],
            marker=style['marker'],
            linewidth=2.2,
            markersize=7,
            label=model,
            zorder=3,
        )

        other_model = next(name for name in MODEL_ORDER if name != model)
        other_values = (
            summary.loc[summary['subject_model'] == other_model]
            .set_index('condition')
            .reindex(CONDITION_ORDER)[column]
            .astype(float)
            .to_numpy()
            * 100
        )
        for xi, value, other_value in zip(x, values, other_values):
            offset = 8 if value >= other_value else -15
            va = 'bottom' if offset > 0 else 'top'
            ax.annotate(
                f'{value:.1f}',
                (xi, value),
                xytext=(0, offset),
                textcoords='offset points',
                ha='center',
                va=va,
                color=style['color'],
                fontsize=8.5,
                fontweight='semibold',
            )

    ax.set_title(f'({panel}) {title}', loc='left', fontweight='semibold')
    ax.set_ylabel(ylabel)
    ax.set_ylim(0, 100)
    ax.set_yticks(np.arange(0, 101, 20))
    ax.grid(axis='y', color='#D9D9D9', linewidth=0.7, alpha=0.8)
    ax.set_axisbelow(True)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

for ax in axes[-1]:
    ax.set_xticks(x)
    ax.set_xticklabels(CONDITION_ORDER)

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(
    handles, labels,
    loc='upper center',
    ncol=2,
    frameon=False,
    bbox_to_anchor=(0.5, 0.98),
)

pdf_path = OUTPUT_DIR / 'report_alignment_diagnostic_conditions.pdf'
png_path = OUTPUT_DIR / 'report_alignment_diagnostic_conditions.png'
fig.savefig(pdf_path, bbox_inches='tight')
fig.savefig(png_path, dpi=300, bbox_inches='tight')
plt.show()

print(f'Saved PDF: {pdf_path}')
print(f'Saved PNG: {png_path}')

## Complete categorical outcomes

This figure retains every conclusion- and uncertainty-alignment category. It therefore shows whether reductions in contradiction or stronger-than-reference uncertainty are accompanied by aligned conclusions, omissions, weaker conclusions, or non-assessable cases.

In [ ]:
row_order = [
    (model, condition)
    for model in MODEL_ORDER
    for condition in CONDITION_ORDER
]
row_index = pd.MultiIndex.from_tuples(
    row_order, names=['subject_model', 'condition']
)
row_labels = [f'{model} - {condition}' for model, condition in row_order]
y_positions = np.array([6.0, 5.0, 4.0, 2.5, 1.5, 0.5])

def categorical_distribution(column, categories):
    counts = (
        reviews.groupby(
            ['subject_model', 'condition', column], observed=True
        ).size()
        .unstack(fill_value=0)
        .reindex(index=row_index, fill_value=0)
        .reindex(columns=categories, fill_value=0)
    )
    return counts.div(counts.sum(axis=1), axis=0) * 100

conclusion_categories = [
    'aligned', 'partially_aligned', 'not_addressed', 'contradicted'
]
uncertainty_categories = [
    'matched', 'stronger_than_reference', 'weaker_than_reference',
    'not_assessable',
]
conclusion_dist = categorical_distribution(
    'conclusion_alignment', conclusion_categories
)
uncertainty_dist = categorical_distribution(
    'uncertainty_alignment', uncertainty_categories
)

category_styles = {
    'aligned': ('Aligned', '#2E8B57', 'white'),
    'partially_aligned': ('Partially aligned', '#E5A522', '#1A1A1A'),
    'not_addressed': ('Not addressed', '#9AA3B1', '#1A1A1A'),
    'contradicted': ('Contradicted', '#C9433D', 'white'),
    'matched': ('Matched', '#2E8B57', 'white'),
    'stronger_than_reference': (
        'Stronger than reference', '#C9433D', 'white'
    ),
    'weaker_than_reference': ('Weaker than reference', '#4C78A8', 'white'),
    'not_assessable': ('Not assessable', '#AAB2BD', '#1A1A1A'),
}

def draw_stacked_panel(ax, distribution, categories, title, show_labels):
    left = np.zeros(len(distribution))
    for category in categories:
        label, color, text_color = category_styles[category]
        values = distribution[category].to_numpy(dtype=float)
        ax.barh(
            y_positions, values, left=left, height=0.72,
            color=color, edgecolor='white', linewidth=0.7, label=label,
        )
        for y, start, value in zip(y_positions, left, values):
            if value >= 7.0:
                ax.text(
                    start + value / 2, y, f'{value:.1f}%',
                    ha='center', va='center', color=text_color,
                    fontsize=7.5, fontweight='semibold',
                )
        left += values

    ax.set_title(title, fontweight='semibold', pad=10)
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 6.7)
    ax.set_xticks(np.arange(0, 101, 20))
    ax.set_xlabel('Reports (%)')
    ax.set_yticks(y_positions)
    if show_labels:
        ax.set_yticklabels(row_labels)
    else:
        ax.tick_params(axis='y', labelleft=False)
    ax.axhline(3.25, color='#BFC5CC', linewidth=0.8)
    ax.grid(axis='x', color='#D9D9D9', linewidth=0.6, alpha=0.65)
    ax.set_axisbelow(True)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.legend(
        loc='upper center', bbox_to_anchor=(0.5, -0.16),
        ncol=2, frameon=False, fontsize=8,
    )

fig, (ax_conclusion, ax_uncertainty) = plt.subplots(
    1, 2, figsize=(12.0, 5.4), sharey=True
)
fig.subplots_adjust(left=0.22, right=0.99, top=0.84, bottom=0.24, wspace=0.06)
fig.suptitle(
    'Categorical outcomes by model and context condition',
    y=0.97, fontsize=12, fontweight='semibold',
)

draw_stacked_panel(
    ax_conclusion, conclusion_dist, conclusion_categories,
    'Conclusion alignment', True,
)
draw_stacked_panel(
    ax_uncertainty, uncertainty_dist, uncertainty_categories,
    'Uncertainty alignment', False,
)

categorical_pdf = OUTPUT_DIR / 'report_alignment_categorical_outcomes.pdf'
categorical_png = OUTPUT_DIR / 'report_alignment_categorical_outcomes.png'
fig.savefig(categorical_pdf, bbox_inches='tight')
fig.savefig(categorical_png, dpi=300, bbox_inches='tight')
plt.show()

print(f'Saved PDF: {categorical_pdf}')
print(f'Saved PNG: {categorical_png}')

## Mutually exclusive key-point outcomes

This supplementary table is useful when interpreting whether lower contradiction is accompanied by greater compatible coverage or by additional omissions. It is not required in the main thesis figure.

In [ ]:
keypoint_status = (
    keypoints.groupby(
        ['subject_model', 'condition', 'status'], observed=True
    ).size()
    .rename('count')
    .reset_index()
)
keypoint_status['proportion'] = (
    keypoint_status['count']
    / keypoint_status.groupby(
        ['subject_model', 'condition'], observed=True
    )['count'].transform('sum')
)
keypoint_status['subject_model'] = pd.Categorical(
    keypoint_status['subject_model'], categories=MODEL_ORDER, ordered=True
)
keypoint_status['condition'] = pd.Categorical(
    keypoint_status['condition'], categories=CONDITION_ORDER, ordered=True
)
keypoint_status_table = (
    keypoint_status.pivot_table(
        index=['subject_model', 'condition'],
        columns='status',
        values='proportion',
        observed=True,
    )
    .reindex(columns=['covered', 'partially_covered', 'missing', 'contradicted'])
    .sort_index()
)
keypoint_status_table.to_csv(
    OUTPUT_DIR / 'three_condition_keypoint_status.csv'
)
display(keypoint_status_table.style.format('{:.1%}'))